# Multi-KB Semantic Routing via AgentCore Gateway

This notebook demonstrates how an agent automatically routes queries to the correct Knowledge Base based on **tool descriptions** when multiple KBs are registered as Gateway targets.

### The field question

> *"When using multiple KBs in a gateway, how does the semantic search route to the correct KB? Is the target description used for routing?"*

### How multi-KB routing works

```
                         ┌─────────────────────────┐
Agent Query              │   AgentCore Gateway     │
"What was the tornado    │                         │
 damage in 2023?"        │  Target A: financial-kb │ ← description: "Financial reports,
         │               │    (Octank 10K)         │    revenue, risk factors"
         │               │                         │
         ▼               │  Target B: weather-kb   │ ← description: "Weather events,
┌──────────────┐         │    (Tornado reports)    │    tornado damage, storms"
│ Strands Agent│────MCP──│                         │
│ (Claude FM)  │         └─────────────────────────┘
└──────────────┘
         │
         ▼
   FM sees tool list:
   - financial-kb_Retrieve: "Retrieve from Octank Financial..."
   - weather-kb_Retrieve: "Retrieve from tornado/weather..."
         │
         ▼
   FM selects: weather-kb_Retrieve ← based on description match
```

### Key insight

**The Gateway does NOT do the routing.** The routing happens at the **agent's FM level**:
1. Gateway exposes each KB as an MCP tool with a `name` + `description`
2. Agent's FM sees all available tools via `tools/list`
3. FM decides which tool to call based on the **tool description** matching the user's query
4. Gateway executes the selected tool against the correct KB

The **target description** you set when creating the target IS the tool description the FM sees. Writing clear, specific descriptions is essential for accurate routing.

### What this notebook demonstrates

1. Create 2 KBs: Financial (Octank 10K) + Weather (Tornado reports)
2. Create a Gateway with 2 targets — each with distinct descriptions
3. Agent queries that should route to specific KBs
4. **Transparent display** of which KB tool was selected for each query

### Prerequisites

- AWS credentials with Bedrock, IAM, and AgentCore permissions
- Model access for Claude and managed embedding
- Python 3.10+
- `strands-agents`, `mcp-proxy-for-aws`

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --quiet
%pip install strands-agents strands-agents-tools mcp-proxy-for-aws --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

In [ ]:
import boto3
import sys
import time
import os
import json
import pprint

sys.path.insert(0, "..")

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# Bucket for both KBs
bucket_name = f'bedrock-bmkb-multikb-{suffix}-{account_id}'

# Model
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:  {region}')
print(f'Account: {account_id}')
print(f'Bucket:  {bucket_name}')
print(f'Model:   {generation_model_arn}')

## Step 2 — Create S3 bucket and upload documents

We upload documents to two separate prefixes to isolate the content:
- `financial/` — Octank Financial 10K report
- `weather/` — Tornado reports

In [ ]:
# Create bucket
try:
    s3_client.head_bucket(Bucket=bucket_name)
except Exception:
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

# Upload Octank Financial 10K → financial/ prefix
s3_client.upload_file('../synthetic_dataset/octank_financial_10K.pdf', bucket_name, 'financial/octank_financial_10K.pdf')
print(f'✓ Uploaded: s3://{bucket_name}/financial/octank_financial_10K.pdf')

# Upload Tornado report → weather/ prefix
s3_client.upload_file('../synthetic_dataset/tornadoes_report.pdf', bucket_name, 'weather/tornadoes_report.pdf')
print(f'✓ Uploaded: s3://{bucket_name}/weather/tornadoes_report.pdf')

## Step 3 — Create two Knowledge Bases

Each KB has:
- Different content (financial vs weather)
- Different S3 prefix (isolated data sources)

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

# KB 1: Financial reports
print('=== Creating Financial KB ===')
kb_financial = ManagedKnowledgeBase(
    kb_name=f'bmkb-financial-{suffix}',
    kb_description='Octank Financial 10K annual reports with revenue, risk factors, and business strategy.',
    data_sources=[{
        'type': 'S3',
        'bucket_name': bucket_name,
        's3_prefix': 'financial/',
    }],
    embedding_model=None,  # Managed default
    enable_logging=True,
    region_name=region,
    suffix=f'fin-{suffix}',
)
print(f'Financial KB ID: {kb_financial.kb_id}')
time.sleep(15)
kb_financial.start_ingestion_job()

print()

# KB 2: Weather/tornado reports
print('=== Creating Weather KB ===')
kb_weather = ManagedKnowledgeBase(
    kb_name=f'bmkb-weather-{suffix}',
    kb_description='Tornado and severe weather reports with storm damage, casualties, and affected regions.',
    data_sources=[{
        'type': 'S3',
        'bucket_name': bucket_name,
        's3_prefix': 'weather/',
    }],
    embedding_model=None,
    enable_logging=True,
    region_name=region,
    suffix=f'wea-{suffix}',
)
print(f'Weather KB ID: {kb_weather.kb_id}')
time.sleep(15)
kb_weather.start_ingestion_job()

print(f'\n✓ Two KBs ready:')
print(f'  Financial: {kb_financial.kb_id}')
print(f'  Weather:   {kb_weather.kb_id}')

## Step 4 — Create Gateway IAM Role

The Gateway needs its own IAM role with `bedrock:Retrieve` on **both** KBs, plus `bedrock:GetKnowledgeBase` and `bedrock:ListKnowledgeBases` for target validation.

In [ ]:
iam_client = boto3.client('iam')
gateway_role_name = f'AmazonBedrockGatewayRole_multikb_{suffix}'

# Trust policy
gw_trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
        'Action': 'sts:AssumeRole',
    }]
}

# Permission policy — Retrieve from BOTH KBs
gw_permission_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Effect': 'Allow',
            'Action': ['bedrock:Retrieve', 'bedrock:GetKnowledgeBase', 'bedrock:ListKnowledgeBases'],
            'Resource': [
                f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/{kb_financial.kb_id}',
                f'arn:aws:bedrock:{region}:{account_id}:knowledge-base/{kb_weather.kb_id}',
            ]
        }
    ]
}

# Create role
try:
    role_resp = iam_client.create_role(
        RoleName=gateway_role_name,
        AssumeRolePolicyDocument=json.dumps(gw_trust_policy),
        Description='Gateway role for multi-KB routing demo',
    )
    print(f'Created role: {gateway_role_name}')
except iam_client.exceptions.EntityAlreadyExistsException:
    role_resp = iam_client.get_role(RoleName=gateway_role_name)
    print(f'Role already exists: {gateway_role_name}')

gateway_role_arn = role_resp['Role']['Arn']

# Create and attach policy
gw_policy_name = f'AmazonBedrockGatewayRetrievePolicy_multikb_{suffix}'
try:
    policy_resp = iam_client.create_policy(
        PolicyName=gw_policy_name,
        PolicyDocument=json.dumps(gw_permission_policy),
    )
    policy_arn = policy_resp['Policy']['Arn']
    print(f'Created policy: {gw_policy_name}')
except iam_client.exceptions.EntityAlreadyExistsException:
    policy_arn = f'arn:aws:iam::{account_id}:policy/{gw_policy_name}'
    print(f'Policy already exists: {gw_policy_name}')

iam_client.attach_role_policy(RoleName=gateway_role_name, PolicyArn=policy_arn)

print(f'\nGateway Role ARN: {gateway_role_arn}')
print('Waiting 30s for IAM propagation...')
time.sleep(30)

## Step 5 — Create Gateway with two KB targets

**The description you set on each target IS the tool description the agent's FM sees.**

Good descriptions for routing:
- ✅ `"Retrieve from Octank Financial 10K reports: revenue, profit, risk factors, business strategy, financial metrics"`
- ✅ `"Retrieve from weather and tornado reports: storm damage, tornado statistics, severe weather events"`

Bad descriptions (too vague for routing):
- ❌ `"Retrieve documents"`
- ❌ `"Knowledge base"`

In [ ]:
# Create Gateway
gateway_name = f'multi-kb-gw-{suffix}'

gateway = kb_financial.create_gateway(
    gateway_name=gateway_name,
    gateway_role_arn=gateway_role_arn,
)

gateway_id = gateway['gateway_id']
gateway_url = gateway['gateway_url']

print(f'Gateway ID:  {gateway_id}')
print(f'Gateway URL: {gateway_url}')

In [ ]:
# Target 1: Financial KB — with specific, routing-friendly description
target_financial = kb_financial.create_gateway_kb_target(
    gateway_id=gateway_id,
    target_name='financial-kb',
    num_results=5,
    description='Octank Financial 10K annual reports: revenue, profit, risk factors, business strategy, growth, and financial metrics.',
)
print(f'Financial target: {target_financial["target_id"]} — {target_financial["status"]}')

# Target 2: Weather KB — with specific, routing-friendly description
target_weather = kb_weather.create_gateway_kb_target(
    gateway_id=gateway_id,
    target_name='weather-kb',
    num_results=5,
    description='Tornado and severe weather reports: storm damage, casualties, property loss, affected states, and natural disaster statistics.',
)
print(f'Weather target:   {target_weather["target_id"]} — {target_weather["status"]}')

print(f'\n✓ Gateway has 2 KB targets. Agents will see 2 tools (one per KB).')

## Step 6 — Verify tool descriptions via MCP tools/list

Show what the agent FM sees when it connects to the Gateway — the tool names and descriptions that drive routing.

In [ ]:
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client
from strands.tools.mcp import MCPClient
import asyncio

mcp_client = MCPClient(lambda: aws_iam_streamablehttp_client(
    endpoint=gateway_url,
    aws_region=region,
    aws_service='bedrock-agentcore',
))

# List tools — this is what the agent FM sees for routing decisions
with mcp_client:
    tools = mcp_client.list_tools_sync()

print(f'=== MCP Tools Discovered ({len(tools)} tools) ===')
print(f'The agent FM uses these descriptions to decide which KB to query.\n')

for tool in tools:
    spec = tool.tool_spec
    print(f'Tool: {tool.tool_name}')
    print(f'  Description: {spec.get("description", "N/A")}')
    print()

## Step 7 — Strands Agent with transparent tool selection

Create an agent that connects to the Gateway. We'll run queries and show **which tool (KB) the agent chose** for each one.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

# Create the agent with Gateway MCP tools
model = BedrockModel(
    model_id=f'{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0',
    region_name=region,
)

agent = Agent(
    model=model,
    tools=[mcp_client],
    system_prompt=(
        'You are a research assistant with access to multiple knowledge bases. '
        'Choose the most appropriate KB tool based on the query topic. '
        'Always cite which knowledge base you used in your response.'
    ),
)

print('Agent created with Gateway MCP tools.')
print(f'Available tools: {[t.tool_name for t in tools]}')

## Step 8 — Test semantic routing

Run queries that should route to different KBs. After each response, we inspect the tool call to show **which KB was selected**.

In [ ]:
# Test queries — designed to route to specific KBs
test_queries = [
    # Should route to FINANCIAL KB
    "What was Octank Financial's total revenue and which business segment grew the fastest?",
    # Should route to WEATHER KB
    "How many tornadoes were reported and what was the total property damage?",
    # Should route to FINANCIAL KB
    "What are the key risk factors mentioned in the annual report?",
    # Should route to WEATHER KB
    "Which states were most affected by severe storms?",
    # Ambiguous — interesting to see which KB the agent picks
    "What were the biggest losses reported?",
]

print('=== Semantic Routing Test ===\n')

for i, query in enumerate(test_queries, 1):
    print(f'--- Query {i} ---')
    print(f'Q: {query}')
    
    # Run the agent
    result = agent(query)
    
    # Extract which tool was called from the agent's conversation
    tool_calls = []
    for msg in agent.messages:
        if hasattr(msg, 'content') and isinstance(msg.get('content'), list):
            for block in msg['content']:
                if isinstance(block, dict) and block.get('toolUse'):
                    tool_calls.append(block['toolUse']['name'])
    
    # Show routing decision
    kb_selected = tool_calls[-1] if tool_calls else 'unknown'
    if 'financial' in kb_selected.lower():
        routing = '📊 FINANCIAL KB'
    elif 'weather' in kb_selected.lower():
        routing = '🌪️ WEATHER KB'
    else:
        routing = f'❓ {kb_selected}'
    
    print(f'Routed to: {routing}')
    print(f'Tool called: {kb_selected}')
    print(f'Answer: {str(result)[:200]}...')
    print()
    
    # Reset conversation for next query
    agent.messages = []

## Step 9 — Description quality comparison

Let's demonstrate why good descriptions matter. The same query with vague vs specific descriptions produces different routing accuracy.

In [ ]:
print('=== Routing Summary ===')
print()
print('Target descriptions used:')
print(f'  financial-kb: "...revenue, profit margins, business segments, financial risk factors..."')
print(f'  weather-kb:   "...tornado damage, storm statistics, weather events, casualties..."')
print()
print('Key findings:')
print('  1. The AGENT FM (not Gateway) makes the routing decision')
print('  2. Tool descriptions are what the FM uses to pick the right KB')
print('  3. Specific, domain-relevant descriptions → accurate routing')
print('  4. Vague descriptions ("Retrieve documents") → random/wrong routing')
print()
print('Best practices for multi-KB routing:')
print('  • Include key topics/domains in the description')
print('  • Use action words: "Use for questions about..."')
print('  • Make descriptions mutually exclusive between targets')
print('  • Avoid generic descriptions like "Retrieve from knowledge base"')

## Step 10 — Cleanup

In [ ]:
# Uncomment to delete all resources
kb_financial.delete_gateway(gateway_id=gateway_id, target_id=target_financial['target_id'])
kb_weather.delete_gateway(gateway_id=gateway_id, target_id=target_weather['target_id'])
kb_financial.delete_kb(delete_iam=True)
kb_weather.delete_kb(delete_iam=True, delete_s3_bucket=True)

## Summary — Answering the Field Question

### KB description vs Target description

| Description | Where it's set | Used for routing? |
|-------------|---------------|-------------------|
| **KB description** | `create_knowledge_base(description=...)` | ❌ No — this is metadata about the KB itself, not propagated to Gateway |
| **Target description** | Set when attaching KB to Gateway (target creation) | ✅ Yes — this becomes the MCP tool description the agent FM sees |

The KB-level description does NOT automatically become the tool description in the Gateway. You must **explicitly set the description on the Gateway target** when attaching the KB.

### How does multi-KB routing work in Gateway?

| Layer | Who decides | Based on what |
|-------|-------------|---------------|
| **Agent FM** | Which tool to call | Tool `description` from `tools/list` |
| **Gateway** | Executes the call | Routes to the correct KB target |

### Is the target description used for routing?

**Yes** — the target `description` becomes the MCP tool description. The agent's FM uses this description to decide which tool/KB to call. The Gateway itself doesn't do semantic matching — it's the FM's tool selection that provides routing.

### Why might routing seem wrong?

| Problem | Cause | Fix |
|---------|-------|-----|
| Agent always picks the same KB | Descriptions too similar or too vague | Write specific, mutually exclusive descriptions |
| Agent picks wrong KB | Description doesn't match the content domain | Update description to reflect actual KB content |
| Agent calls all KBs | Agent thinks it needs info from both | Add "ONLY use for..." phrasing in description |

### Best practice: description template

```
"Retrieve from [DOMAIN] knowledge base. Use ONLY for questions about [TOPIC 1], [TOPIC 2], [TOPIC 3]. Contains [DATA TYPE] from [SOURCE]."
```

Examples:
- `"Retrieve from financial knowledge base. Use ONLY for questions about revenue, profit, risk factors, business strategy. Contains Octank Financial 10K annual reports."`
- `"Retrieve from weather knowledge base. Use ONLY for questions about tornadoes, storms, property damage, weather events. Contains NOAA severe weather reports."`

### Key takeaway

The routing quality is only as good as your tool descriptions. The Gateway GA behavior is correct — descriptions ARE used for routing, but the routing logic lives in the agent's FM, not in the Gateway itself. Writing specific, non-overlapping descriptions is the key to accurate multi-KB semantic routing.